# Предобработка данных для анализа рыночных корзин

**Задача:** подготовить транзакционные данные для поиска ассоциативных правил. \
**Источник:** [Kaggle: Market Basket Analysis](https://www.kaggle.com/datasets/aslanahmedov/market-basket-analysis) \
**Результат:** `data/processed.parquet` — 386103 транзакции, 7 колонок, 38.5 MB.

In [87]:
import kagglehub
from pathlib import Path
import shutil


dataset_path = Path("./data/Assignment-1_Data.csv")

if dataset_path.exists():
    print("File exists")
else:
    path = kagglehub.dataset_download("aslanahmedov/market-basket-analysis")
    shutil.move(path, './data')

File exists


In [88]:
import pandas as pd
import numpy as np


df = pd.read_csv("./data/Assignment-1_Data.csv", on_bad_lines='skip', low_memory=False, sep=';')
df.head()

,BillNo,Itemname,Quantity,Date,Price,CustomerID,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,01.12.2010 08:26,"2,55",17850.0,United Kingdom
1,536365,WHITE METAL LANTERN,6,01.12.2010 08:26,"3,39",17850.0,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,01.12.2010 08:26,"2,75",17850.0,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,01.12.2010 08:26,"3,39",17850.0,United Kingdom
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,01.12.2010 08:26,"3,39",17850.0,United Kingdom


In [89]:
df.shape

(522064, 7)

Вес датафрейма можно сократить, если заменить тип `object` на более подходящий для колонки

In [90]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522064 entries, 0 to 522063
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   BillNo      522064 non-null  object 
 1   Itemname    520609 non-null  object 
 2   Quantity    522064 non-null  int64  
 3   Date        522064 non-null  object 
 4   Price       522064 non-null  object 
 5   CustomerID  388023 non-null  float64
 6   Country     522064 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 162.7 MB


Колонка BillNo содержала значения, не являющиеся 6-значными числами, для однородности данных их следует удалить. Если поменять тип на `np.int32` то можно сократить объем занимаемой памяти.

In [91]:
indexes_to_drop = df[~df['BillNo'].str.isdigit()].index      
df.drop(indexes_to_drop, inplace=True)

df['BillNo'] = df['BillNo'].astype(np.int32)
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 522061 entries, 0 to 522063
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   BillNo      522061 non-null  int32  
 1   Itemname    520606 non-null  object 
 2   Quantity    522061 non-null  int64  
 3   Date        522061 non-null  object 
 4   Price       522061 non-null  object 
 5   CustomerID  388023 non-null  float64
 6   Country     522061 non-null  object 
dtypes: float64(1), int32(1), int64(1), object(4)
memory usage: 141.3 MB


Колонка CustomerID содержит пропущенные значения. Удалить только строки с пропущенным значением айди будет недостаточно т.к. тогда некоторые транзакции будут неполными, обрывистыми. Поэтому нужно удалять именно транзакции, где есть какие-либо недостатки.

In [92]:
display(df[df['CustomerID'] > 65535].sum().sum(), df['CustomerID'].isna().sum())

np.float64(0.0)

np.int64(134038)

In [93]:
bad_transcations = df[df['CustomerID'].isna()]['BillNo'].unique()
indexes_to_drop = df[df['BillNo'].isin(bad_transcations)].index
df.drop(indexes_to_drop, inplace=True)
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 388023 entries, 0 to 522063
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   BillNo      388023 non-null  int32  
 1   Itemname    388023 non-null  object 
 2   Quantity    388023 non-null  int64  
 3   Date        388023 non-null  object 
 4   Price       388023 non-null  object 
 5   CustomerID  388023 non-null  float64
 6   Country     388023 non-null  object 
dtypes: float64(1), int32(1), int64(1), object(4)
memory usage: 105.0 MB


In [94]:
df['CustomerID'] = df['CustomerID'].astype(np.int16)
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 388023 entries, 0 to 522063
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   BillNo      388023 non-null  int32 
 1   Itemname    388023 non-null  object
 2   Quantity    388023 non-null  int64 
 3   Date        388023 non-null  object
 4   Price       388023 non-null  object
 5   CustomerID  388023 non-null  int16 
 6   Country     388023 non-null  object
dtypes: int16(1), int32(1), int64(1), object(4)
memory usage: 102.8 MB


Для удобства дальнейшей работы и экономии памяти колонку Price выгодно привести к типу `np.float32`.

In [95]:
df['Price'] = df['Price'].str.replace(',', '.')
df['Price'] = df['Price'].astype(np.float32)
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 388023 entries, 0 to 522063
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   BillNo      388023 non-null  int32  
 1   Itemname    388023 non-null  object 
 2   Quantity    388023 non-null  int64  
 3   Date        388023 non-null  object 
 4   Price       388023 non-null  float32
 5   CustomerID  388023 non-null  int16  
 6   Country     388023 non-null  object 
dtypes: float32(1), int16(1), int32(1), int64(1), object(3)
memory usage: 84.7 MB


Существуют товары с ценой 0 - их тоже необходимо удалить.

In [96]:
df.describe()

,BillNo,Quantity,Price,CustomerID
count,388023.000000,388023.000000,388023.000000,388023.000000
mean,560610.618886,12.892140,3.079257,15316.931710
std,13127.766961,182.605318,21.965773,1721.846964
min,536365.000000,1.000000,0.000000,12346.000000
25%,549225.000000,2.000000,1.250000,13950.000000
50%,561888.000000,5.000000,1.950000,15265.000000
75%,572131.000000,12.000000,3.750000,16837.000000
max,581587.000000,80995.000000,8142.750000,18287.000000


In [97]:
bad_transcations = df[df['Price'] == 0]['BillNo'].unique()
indexes_to_drop = df[df['BillNo'].isin(bad_transcations)].index
df.drop(indexes_to_drop, inplace=True)
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 387465 entries, 0 to 522063
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   BillNo      387465 non-null  int32  
 1   Itemname    387465 non-null  object 
 2   Quantity    387465 non-null  int64  
 3   Date        387465 non-null  object 
 4   Price       387465 non-null  float32
 5   CustomerID  387465 non-null  int16  
 6   Country     387465 non-null  object 
dtypes: float32(1), int16(1), int32(1), int64(1), object(3)
memory usage: 84.6 MB


Перевод колонки Date к типу `datetime` экономит память и может пригодиться в последующей работе.

In [98]:
df['Date'] = pd.to_datetime(df['Date'], format='%d.%m.%Y %H:%M')
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 387465 entries, 0 to 522063
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   BillNo      387465 non-null  int32         
 1   Itemname    387465 non-null  object        
 2   Quantity    387465 non-null  int64         
 3   Date        387465 non-null  datetime64[ns]
 4   Price       387465 non-null  float32       
 5   CustomerID  387465 non-null  int16         
 6   Country     387465 non-null  object        
dtypes: datetime64[ns](1), float32(1), int16(1), int32(1), int64(1), object(2)
memory usage: 63.5 MB


In [99]:
df.head()

,BillNo,Itemname,Quantity,Date,Price,CustomerID,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


Удаление "мёртвых чеков", по которым нельзя построить ассоциативные правила.

In [100]:
bad_transcations = df.groupby('BillNo').filter(lambda x: x['Itemname'].count() <= 1)
indexes_to_drop = bad_transcations.index
df.drop(indexes_to_drop, inplace=True)
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 386103 entries, 0 to 522063
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   BillNo      386103 non-null  int32         
 1   Itemname    386103 non-null  object        
 2   Quantity    386103 non-null  int64         
 3   Date        386103 non-null  datetime64[ns]
 4   Price       386103 non-null  float32       
 5   CustomerID  386103 non-null  int16         
 6   Country     386103 non-null  object        
dtypes: datetime64[ns](1), float32(1), int16(1), int32(1), int64(1), object(2)
memory usage: 63.3 MB


In [101]:
df['Country'].unique().shape

(28,)

In [102]:
df['Country'] = df['Country'].astype('category')
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 386103 entries, 0 to 522063
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   BillNo      386103 non-null  int32         
 1   Itemname    386103 non-null  object        
 2   Quantity    386103 non-null  int64         
 3   Date        386103 non-null  datetime64[ns]
 4   Price       386103 non-null  float32       
 5   CustomerID  386103 non-null  int16         
 6   Country     386103 non-null  category      
dtypes: category(1), datetime64[ns](1), float32(1), int16(1), int32(1), int64(1), object(1)
memory usage: 40.7 MB


In [103]:
df.describe()

,BillNo,Quantity,Date,Price,CustomerID
count,386103.000000,386103.000000,386103,386103.000000,386103.000000
mean,560621.202335,12.087277,2011-07-10 23:48:53.825844224,2.953092,15318.660148
min,536365.000000,1.000000,2010-12-01 08:26:00,0.001000,12347.000000
25%,549240.000000,2.000000,2011-04-07 11:37:00,1.250000,13956.000000
50%,561893.000000,5.000000,2011-07-31 14:39:00,1.950000,15270.000000
75%,572160.500000,12.000000,2011-10-21 10:25:30,3.750000,16839.000000
max,581587.000000,4800.000000,2011-12-09 12:50:00,3949.320068,18287.000000
std,13122.275984,38.012533,NaN,11.097044,1720.823951


Quantity имеет максимум 4800 после очистки — влезает в `np.int16`, экономим память.

In [104]:
df['Quantity'] = df['Quantity'].astype(np.int16)
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 386103 entries, 0 to 522063
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   BillNo      386103 non-null  int32         
 1   Itemname    386103 non-null  object        
 2   Quantity    386103 non-null  int16         
 3   Date        386103 non-null  datetime64[ns]
 4   Price       386103 non-null  float32       
 5   CustomerID  386103 non-null  int16         
 6   Country     386103 non-null  category      
dtypes: category(1), datetime64[ns](1), float32(1), int16(2), int32(1), object(1)
memory usage: 38.5 MB


Из-за пробелов в конце строки или в начале, двойных пробелов - одинаковые строки могут считаться разными. 

In [105]:
df['Itemname'].isna().sum()

np.int64(0)

In [106]:
df['Itemname'].str.startswith(' ').sum()

np.int64(0)

In [107]:
df['Itemname'].str.endswith(' ').sum()

np.int64(0)

In [108]:
(df['Itemname'].str.find('  ') > 0).sum()

np.int64(13951)

In [109]:
df['Itemname'] = df['Itemname'].str.replace(r'\s+', ' ', regex=True)
(df['Itemname'].str.find('  ') > 0).sum()

np.int64(0)

In [110]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 386103 entries, 0 to 522063
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   BillNo      386103 non-null  int32         
 1   Itemname    386103 non-null  object        
 2   Quantity    386103 non-null  int16         
 3   Date        386103 non-null  datetime64[ns]
 4   Price       386103 non-null  float32       
 5   CustomerID  386103 non-null  int16         
 6   Country     386103 non-null  category      
dtypes: category(1), datetime64[ns](1), float32(1), int16(2), int32(1), object(1)
memory usage: 38.5 MB


In [111]:
df.shape

(386103, 7)

In [112]:
df.to_parquet('./data/processed.parquet', index=False)

Этот формат данных сохраняет типы колонок и весит меньше, чем pkl:

In [113]:
df_check = pd.read_parquet('./data/processed.parquet')
print(f"Shape: {df_check.shape}")
print(f"Memory: {df_check.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"Dtypes:\n{df_check.dtypes}")

Shape: (386103, 7)
Memory: 35.5 MB
Dtypes:
BillNo                 int32
Itemname              object
Quantity               int16
Date          datetime64[ns]
Price                float32
CustomerID             int16
Country             category
dtype: object


## Итог
- Удалено 135 961 строка (26%) — "битые" чеки, нулевые цены, одиночные транзакции.
- Оптимизированы типы.
- Память снижена с 162.7 МБ до 38.5 МБ (−76%).
- Результат сохранён в `data/processed.parquet`.